In [ ]:
import sys
import os
import random
import shutil
import yaml
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
from sklearn.model_selection import train_test_split

import torch
from ultralytics import YOLO
import cv2

random.seed(42)
np.random.seed(42)

In [ ]:
BASE_DIR = Path("/mnt/e/NCSU/Fall_2025/Board Game/board_segmentation")
DATASET_DIR = BASE_DIR / "board_segmentation_dataset"
IMAGES_DIR = DATASET_DIR / "images_aug"
LABELS_DIR = DATASET_DIR / "labels_aug"
YOLO_DATASET_DIR = BASE_DIR / "yolo_segmentation_dataset"

print(f"Number of images: {len(list(IMAGES_DIR.glob('*.jpg')))}")
print(f"Number of labels: {len(list(LABELS_DIR.glob('*.txt')))}")

classes_file = DATASET_DIR / "classes.txt"
with open(classes_file, 'r') as f:
    classes = [line.strip() for line in f.readlines() if line.strip()]
print(f"Classes: {classes}")

In [ ]:
image_files = list(IMAGES_DIR.glob("*.jpg"))
image_names = [img.stem for img in image_files]

print(f"Total images: {len(image_names)}")

valid_pairs = []
for img_name in image_names:
    label_file = LABELS_DIR / f"{img_name}.txt"
    if label_file.exists():
        valid_pairs.append(img_name)
    else:
        print(f"Warning: Label file missing for {img_name}")

print(f"Valid image-label pairs: {len(valid_pairs)}")

In [ ]:
train_names, temp_names = train_test_split(valid_pairs, test_size=0.3, random_state=42)

val_names, test_names = train_test_split(temp_names, test_size=0.333, random_state=42)

print(f"Dataset split:")
print(f"Train: {len(train_names)} ({len(train_names)/len(valid_pairs)*100:.1f}%)")
print(f"Val: {len(val_names)} ({len(val_names)/len(valid_pairs)*100:.1f}%)")
print(f"Test: {len(test_names)} ({len(test_names)/len(valid_pairs)*100:.1f}%)")
print(f"Total: {len(train_names) + len(test_names) + len(val_names)}")

In [ ]:
def create_yolo_structure():
    if YOLO_DATASET_DIR.exists():
        shutil.rmtree(YOLO_DATASET_DIR)

    for split in ['train', 'test', 'val']:
        (YOLO_DATASET_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
        (YOLO_DATASET_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)
    
    print("Created YOLO dataset directory structure.")

create_yolo_structure()

In [ ]:
def copy_files_to_splits():
    
    splits = {
        'train': train_names,
        'test': test_names,
        'val': val_names
    }
    
    for split_name, file_names in splits.items():
        
        for i, file_name in enumerate(file_names):
            src_img = IMAGES_DIR / f"{file_name}.jpg"
            dst_img = YOLO_DATASET_DIR / split_name / 'images' / f"{file_name}.jpg"
            shutil.copy2(src_img, dst_img)

            src_label = LABELS_DIR / f"{file_name}.txt"
            dst_label = YOLO_DATASET_DIR / split_name / 'labels' / f"{file_name}.txt"
            shutil.copy2(src_label, dst_label)
            
            if (i + 1) == len(file_names):
                print(f"Copied {i + 1}/{len(file_names)} files")

    for split_name in splits.keys():
        img_count = len(list((YOLO_DATASET_DIR / split_name / 'images').glob('*.jpg')))
        label_count = len(list((YOLO_DATASET_DIR / split_name / 'labels').glob('*.txt')))
        print(f"{split_name}: {img_count} images, {label_count} labels")

copy_files_to_splits()

In [ ]:
def create_yolo_config():
    
    config = {
        'path': str(YOLO_DATASET_DIR.absolute()),
        'train': 'train/images',
        'val': 'val/images',
        'test': 'test/images',
        'names': {i: name for i, name in enumerate(classes)},
        'nc': len(classes)
    }
    
    config_file = YOLO_DATASET_DIR / 'dataset.yaml'
    with open(config_file, 'w') as f:
        yaml.dump(config, f, default_flow_style=False, sort_keys=False)
    
    print(f"Created YOLO configuration file: {config_file}")
    
    return config_file

config_file = create_yolo_config()

In [ ]:
def plot_segmentation_sample(img_path, label_path, ax):
    img = Image.open(img_path)
    img_array = np.array(img)
    ax.imshow(img_array)

    with open(label_path, 'r') as f:
        lines = f.readlines()
    
    for line in lines:
        parts = line.strip().split()
        if len(parts) >= 6:
            class_id = int(parts[0])
            coords = [float(x) for x in parts[1:]]
            
            # Convert normalized coordinates to pixel coordinates
            h, w = img_array.shape[:2]
            pixel_coords = []
            for i in range(0, len(coords), 2):
                x = coords[i] * w
                y = coords[i+1] * h
                pixel_coords.extend([x, y])
            
            # Create polygon points
            polygon_points = []
            for i in range(0, len(pixel_coords), 2):
                polygon_points.append([pixel_coords[i], pixel_coords[i+1]])
            
            # Plot polygon
            if len(polygon_points) >= 3:
                polygon = plt.Polygon(polygon_points, fill=False, edgecolor='red', linewidth=2)
                ax.add_patch(polygon)
                
                # Add class label
                ax.text(polygon_points[0][0], polygon_points[0][1], 
                       f'Board', color='red', fontsize=12, fontweight='bold',
                       bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
    
    ax.set_title(f"Sample: {img_path.name}")
    ax.axis('off')

# Plot sample images from train set
sample_names = random.sample(train_names, min(4, len(train_names)))

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

for i, sample_name in enumerate(sample_names):
    img_path = YOLO_DATASET_DIR / 'train' / 'images' / f"{sample_name}.jpg"
    label_path = YOLO_DATASET_DIR / 'train' / 'labels' / f"{sample_name}.txt"
    
    plot_segmentation_sample(img_path, label_path, axes[i])

plt.tight_layout()
plt.show()

print("Sample visualization completed. Red polygons show board segmentation annotations.")

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = YOLO('yolov8m-seg.pt')

print("YOLOv8 segmentation model loaded successfully!")
print(f"Model architecture: {model.model}")

model.info()

In [ ]:
EPOCHS = 40
IMAGE_SIZE = 320
BATCH_SIZE = 6
LEARNING_RATE = 0.01

BASE_DIR = Path("/mnt/e/NCSU/Fall_2025/Board Game/board_segmentation")
TRAINING_OUTPUT_DIR = BASE_DIR / "runs" / "try1"
TRAINING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
required_vars = ['config_file', 'model', 'device']
for var in required_vars:
    if var not in globals():
        print(f"ERROR: {var} not defined. Please run previous cells first.")
        raise NameError(f"{var} is not defined")

try:
    print("Starting training...")
    
    results = model.train(
        data=str(config_file),
        epochs=EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        lr0=LEARNING_RATE,
        device='cpu',
        project=str(TRAINING_OUTPUT_DIR.parent),
        name='board_segmentation_80',
        exist_ok=True,
        save_period=5,
        verbose=True,
        plots=False,
        val=True,
        save=True,
        cache=False,
        workers=1,
        optimizer='SGD',
        close_mosaic=5,
        amp=False,
        single_cls=True,
    )

    print(f"Model saved to: {results.save_dir}")

    
except Exception as e:
    print(f"TRAINING FAILED: {e}")

In [ ]:
runs_dir = BASE_DIR / "runs" / "board_segmentation_80"
model_path = runs_dir / "weights" / "best.pt"
best_model = YOLO(str(model_path))
best_model.to('cuda')

test_image_path = YOLO_DATASET_DIR / 'test' / 'images' / f"{test_names[0]}.jpg"
if test_image_path.exists():
    try:
        test_result = best_model(str(test_image_path), verbose=False)
        if len(test_result) > 0 and test_result[0].boxes is not None:
            print("Model inference test PASSED - model is working!")
            inference_success = True
        else:
            print("Model inference test shows no detections - may need more training")
            inference_success = True
    except Exception as e:
        print(f"Model inference test failed: {e}")
        inference_success = False

In [ ]:
import cv2

# Test the model on sample images with better error handling
try:
    # Make sure we have a model loaded
    if 'best_model' not in globals():
        print("❌ No model available. Please run the previous cell first.")
        raise NameError("best_model not defined")
    
    # Test inference on sample images
    print("🔄 Testing model inference on sample images...")
    sample_names = random.sample(test_names, min(4, len(test_names)))

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()

    for i, sample_name in enumerate(sample_names):
        img_path = YOLO_DATASET_DIR / 'test' / 'images' / f"{sample_name}.jpg"
        
        if not img_path.exists():
            print(f"⚠️  Image not found: {img_path}")
            axes[i].text(0.5, 0.5, f"Image not found:\n{sample_name}", 
                        ha='center', va='center', transform=axes[i].transAxes)
            axes[i].axis('off')
            continue
        
        try:
            # Run inference
            results_inference = best_model(str(img_path), verbose=False)
            
            # Get the result for this image
            result = results_inference[0]
            
            # Load original image
            img = Image.open(img_path)
            img_array = np.array(img)
            
            axes[i].imshow(img_array)
            
            # Draw predictions if available
            if result.masks is not None and len(result.masks) > 0:
                masks = result.masks.data.cpu().numpy()
                boxes = result.boxes.xyxy.cpu().numpy()
                scores = result.boxes.conf.cpu().numpy()
                
                for j, (mask, box, score) in enumerate(zip(masks, boxes, scores)):
                    if score > 0.3:  # Lower threshold for testing
                        # Resize mask to image size
                        mask_resized = cv2.resize(mask, (img_array.shape[1], img_array.shape[0]))
                        
                        # Create colored mask overlay
                        colored_mask = np.zeros_like(img_array)
                        colored_mask[mask_resized > 0.5] = [255, 0, 0]  # Red color
                        
                        # Apply mask with transparency
                        axes[i].imshow(colored_mask, alpha=0.3)
                        
                        # Draw bounding box
                        x1, y1, x2, y2 = box
                        rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, 
                                           fill=False, edgecolor='green', linewidth=2)
                        axes[i].add_patch(rect)
                        
                        # Add confidence score
                        axes[i].text(x1, y1-10, f'Board: {score:.2f}', 
                                   color='green', fontsize=10, fontweight='bold',
                                   bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
            else:
                # No detections
                axes[i].text(0.02, 0.98, 'No detections', transform=axes[i].transAxes,
                           verticalalignment='top', fontsize=10, color='red',
                           bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
            
            axes[i].set_title(f"Test: {sample_name}")
            axes[i].axis('off')
            
        except Exception as e:
            print(f"⚠️  Error processing {sample_name}: {e}")
            axes[i].text(0.5, 0.5, f"Error processing:\n{sample_name}", 
                        ha='center', va='center', transform=axes[i].transAxes, color='red')
            axes[i].axis('off')

    plt.tight_layout()
    plt.show()

    print("✅ Inference test completed!")
    print("🟢 Green boxes = detected boards")
    print("🔴 Red overlay = segmentation masks")
    print("📝 If you see 'No detections', the model needs more training or better data")

except Exception as e:
    print(f"❌ Error during inference testing: {e}")
    print("Make sure the model evaluation cell ran successfully first.")

In [ ]:
def mask_to_bbox(mask: np.ndarray):
    """
    Converts a binary segmentation mask to rotated bounding box coordinates.
    Returns 4 corner points of the minimum rotated rectangle
    """
    # Ensure binary mask (values 0 or 1)
    mask_binary = (mask > 0.5).astype(np.uint8) * 255
    
    # Find contours
    contours, _ = cv2.findContours(mask_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        return None
    
    # Get the largest contour
    largest_contour = max(contours, key=cv2.contourArea)
    
    # Get minimum rotated rectangle
    rect = cv2.minAreaRect(largest_contour)
    
    # Get 4 corner points of the rotated rectangle
    box_points = cv2.boxPoints(rect)
    box_points = np.intp(box_points)  # Use np.intp instead of deprecated np.int0
    
    return box_points

In [ ]:
import cv2

# Test the model on sample images with better error handling
try:
    # Make sure we have a model loaded
    if 'best_model' not in globals():
        print("❌ No model available. Please run the previous cell first.")
        raise NameError("best_model not defined")
    
    # Test inference on sample images
    print("🔄 Testing model inference on sample images...")
    sample_names = random.sample(test_names, min(4, len(test_names)))

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()

    for i, sample_name in enumerate(sample_names):
        img_path = YOLO_DATASET_DIR / 'test' / 'images' / f"{sample_name}.jpg"
        
        if not img_path.exists():
            print(f"⚠️  Image not found: {img_path}")
            axes[i].text(0.5, 0.5, f"Image not found:\n{sample_name}", 
                        ha='center', va='center', transform=axes[i].transAxes)
            axes[i].axis('off')
            continue
        
        try:
            # Run inference
            results_inference = best_model(str(img_path), verbose=False)
            
            # Get the result for this image
            result = results_inference[0]
            
            # Load original image
            img = Image.open(img_path)
            img_array = np.array(img)
            
            axes[i].imshow(img_array)
            
            # Draw predictions if available
            if result.masks is not None and len(result.masks) > 0:
                masks = result.masks.data.cpu().numpy()
                
                for j, mask in enumerate(masks):
                    if score > 0.3:  # Lower threshold for testing
                        # Resize mask to image size
                        mask_resized = cv2.resize(mask, (img_array.shape[1], img_array.shape[0]))

                        box_points = mask_to_bbox(mask_resized)
                        
                        if box_points is not None:
                            # Create colored mask overlay
                            colored_mask = np.zeros_like(img_array)
                            colored_mask[mask_resized > 0.5] = [255, 0, 0]  # Red color
                            axes[i].imshow(colored_mask, alpha=0.3)
                            
                            # Draw rotated rectangle using polygon
                            polygon = plt.Polygon(box_points, fill=False, edgecolor='green', linewidth=2)
                            axes[i].add_patch(polygon)
                            
                            # Add confidence score at center
                            center_x = int(np.mean(box_points[:, 0]))
                            center_y = int(np.mean(box_points[:, 1]))
                            axes[i].text(center_x, center_y-10, f'Board: {score:.2f}', 
                                       color='green', fontsize=10, fontweight='bold',
                                       bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
                            
                            print(f"  Sample {i+1}, Mask {j}: bbox = {box_points.tolist()}")
                        
                        # # Create colored mask overlay
                        # colored_mask = np.zeros_like(img_array)
                        # colored_mask[mask_resized > 0.5] = [255, 0, 0]  # Red color
                        
                        # # Apply mask with transparency
                        # axes[i].imshow(colored_mask, alpha=0.3)
                        
                        # # Draw bounding box
                        # x1, y1, x2, y2 = mask_to_bbox(mask_resized)
                        # rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, 
                        #                    fill=False, edgecolor='green', linewidth=2)
                        # axes[i].add_patch(rect)
                        
                        # Add confidence score
                        # axes[i].text(x1, y1-10, f'Board: {score:.2f}', 
                        #            color='green', fontsize=10, fontweight='bold',
                        #            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
            else:
                # No detections
                axes[i].text(0.02, 0.98, 'No detections', transform=axes[i].transAxes,
                           verticalalignment='top', fontsize=10, color='red',
                           bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
            
            axes[i].set_title(f"Test: {sample_name}")
            axes[i].axis('off')
            
        except Exception as e:
            print(f"⚠️  Error processing {sample_name}: {e}")
            axes[i].text(0.5, 0.5, f"Error processing:\n{sample_name}", 
                        ha='center', va='center', transform=axes[i].transAxes, color='red')
            axes[i].axis('off')

    plt.tight_layout()
    plt.show()

    print("✅ Inference test completed!")
    print("🟢 Green boxes = detected boards")
    print("🔴 Red overlay = segmentation masks")
    print("📝 If you see 'No detections', the model needs more training or better data")

except Exception as e:
    print(f"❌ Error during inference testing: {e}")
    print("Make sure the model evaluation cell ran successfully first.")